# GOES GLM lightning over True Color

The Geostationary Lightning Mapper (GLM) reports individual optical **flashes**
with a latitude, longitude and time. Here they are drawn over a True Color image
built from the **Full Disk** scan.

The example is a line of afternoon thunderstorms over the central United States
on **3 October 2023, 20:00 UTC** (about 14:00 local, so there is daylight for
True Color). GOES-16 is used because that region is in the eastern satellite's
view; the code is identical for GOES-18.


## Environment

Everything runs in the environment described by
[`environment.yml`](../environment.yml) at the repository root:

```bash
conda env create -f environment.yml
conda activate goes-viirs
python -m jupyter lab
```


## Setup

In [ ]:
import sys
import warnings
from datetime import datetime
from pathlib import Path

from IPython.display import Image, display
from satpy import Scene
from satpy.utils import PerformanceWarning

warnings.filterwarnings("ignore", category=PerformanceWarning)
warnings.filterwarnings("ignore", message="Mean of empty slice", category=RuntimeWarning)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from examples.domains import DOMAINS
from examples.glm import abi_scan_keys, download_abi, download_lcfa, lcfa_keys, read_flashes
from examples.render_satellite import crop_and_resample_scene, save_map


## 1. Get the data

Two products for the same moment: the ABI Full Disk channels for True Color,
and the GLM flashes for a ten-minute window around the scan.


In [ ]:
BUCKET = "noaa-goes16"
WHEN = datetime(2023, 10, 3, 20, 0)
TRUE_COLOR_CHANNELS = ("C01", "C02", "C03")

DATA_DIR = REPO_ROOT / "data" / "glm-demo"

abi_keys = abi_scan_keys(BUCKET, "ABI-L1b-RadF", WHEN, TRUE_COLOR_CHANNELS)
abi_files = download_abi(BUCKET, abi_keys, DATA_DIR / "abi16f")
print(f"ABI Full Disk files: {len(abi_files)}")
for name in abi_files:
    print("  ", Path(name).name)

glm_files = download_lcfa(BUCKET, lcfa_keys(BUCKET, WHEN, minutes=10), DATA_DIR / "glm16")
print(f"\nGLM LCFA files (10 minutes): {len(glm_files)}")


## 2. Choose a domain

GLM sees the whole disk, so pick the box around the storms.


In [ ]:
DOMAIN_NAME = "plains_storms"
DOMAIN = DOMAINS[DOMAIN_NAME]
RESOLUTION = 0.01

lons, lats = read_flashes(glm_files, bbox=DOMAIN)
print(f"{DOMAIN_NAME}: {DOMAIN}")
print(f"{len(lons)} flashes inside the domain")


## 3. True Color from the Full Disk, cropped to the domain

In [ ]:
scene = Scene(reader="abi_l1b", filenames=abi_files)
scene.load(["true_color"], generate=True)
cropped = crop_and_resample_scene(scene, domain=DOMAIN, resolution=RESOLUTION)
print("composite ready")


## 4. Draw the flashes over the image

Each circle is one GLM flash. They fall on the bright convective cloud tops,
which is the check that the geolocation lines up.


In [ ]:
out = REPO_ROOT / "output" / "glm_true_color.png"
save_map(
    cropped, "true_color", out,
    title="G16 - True Color + GLM flashes",
    points=(lons, lats),
    points_label=f"GLM flashes ({len(lons)})",
    mark_shishaldin=False,
)
display(Image(filename=str(out)))


## Notes

* `minutes=` on `lcfa_keys` sets how long a window of flashes is drawn. A longer
  window shows more of the storm's history in one frame.
* GLM detects the optical flash at cloud top, so flashes appear over the anvil
  rather than at the ground strike point.
* Detection efficiency drops towards the edge of the disk and at high latitude,
  which matters for volcanic lightning at Alaskan latitudes.
